# Week 10

This notebook keeps the assignment subset constraint (items in both train and test), then improves model quality with:
- Per-user validation split from train
- TF-IDF hyperparameter tuning
- Feature block weight tuning
- Final one-shot test evaluation with the best validation configuration

In [3]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, hstack, diags
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from sklearn.metrics import mean_squared_error, mean_absolute_error
import random
from itertools import product

In [2]:
# Load data
train_df = pd.read_csv('csv_files/preprocessed_train_video_games.csv')
test_df = pd.read_csv('csv_files/preprocessed_test_video_games.csv')
metadata_df = pd.read_parquet('datasets/meta_video_games.parquet')

print('train rows:', len(train_df))
print('test rows:', len(test_df))
print('metadata rows:', len(metadata_df))

train rows: 26580
test rows: 6645
metadata rows: 137269


### Subset Constraint
We intentionally keep only items that appear in both train and test

In [4]:
# Keep subset of common items only (assignment constraint)
common_items = set(train_df['item_id']).intersection(set(test_df['item_id']))
metadata_df = metadata_df[metadata_df['item_id'].isin(common_items)].copy()

print('common items:', len(common_items))
print('metadata rows after subset:', len(metadata_df))

common items: 927
metadata rows after subset: 927


In [5]:
import evals_helper as eh 

# Text cleaning helper function
for col in ['description', 'features', 'categories']:
    metadata_df = eh.preprocess_text_column(metadata_df, col)

metadata_df[['item_id', 'description', 'features', 'categories']].head(2)

,item_id,description,features,categories
864,B00000JRSB,amazoncom long recognized as roleplaying games...,1 player rpg 3 disc set excellent graphics so...,video games legacy systems playstation systems...
1312,B00001X50M,product description you are snake a governmen...,lightly armed and facing an army of foes snake...,video games legacy systems playstation systems...


### Validation Protocol
We split only the training interactions into:
- Sub-train interactions: build user profiles
- Validation interactions: tune hyperparameters

Strategy: per-user leave-one-out (when user has at least 2 interactions).

##### Defining the resuable functions

In [6]:
def user_leave_one_out_split(df, user_col='user_id', seed=42):
    rng = np.random.default_rng(seed)
    val_indices = []
    for _, g in df.groupby(user_col):
        if len(g) >= 2:
            val_indices.append(rng.choice(g.index.values, 1)[0])

    val_df = df.loc[val_indices].copy()
    sub_train_df = df.drop(index=val_indices).copy()
    return sub_train_df, val_df

sub_train_df, val_df = user_leave_one_out_split(train_df, user_col='user_id', seed=42)

print('sub-train rows:', len(sub_train_df))
print('validation rows:', len(val_df))
print('users in train:', train_df['user_id'].nunique())
print('users in val:', val_df['user_id'].nunique())

sub-train rows: 25191
validation rows: 1389
users in train: 1389
users in val: 1389


In [7]:
# Build matrices for one configuration
def build_matrices(metadata_base, interactions_df, tfidf_cfg, block_weights):
    w_desc, w_feat, w_cat, w_rating = block_weights

    # TF-IDF for each text block
    v_desc = TfidfVectorizer(stop_words='english', **tfidf_cfg)
    X_desc = v_desc.fit_transform(metadata_base['description'])

    v_feat = TfidfVectorizer(stop_words='english', **tfidf_cfg)
    X_feat = v_feat.fit_transform(metadata_base['features'])

    v_cat = TfidfVectorizer(stop_words='english', **tfidf_cfg)
    X_cat = v_cat.fit_transform(metadata_base['categories'])

    # Numeric feature from interactions used to build this model
    avg_ratings = (interactions_df.groupby('item_id', as_index=False)['rating'].mean().rename(columns={'rating': 'avg_item_rating'}))

    m = metadata_base[['item_id', 'description', 'features', 'categories']].copy()
    m = m.merge(avg_ratings, on='item_id', how='left')
    m['avg_item_rating'] = m['avg_item_rating'].fillna(0.0)
    X_rating = csr_matrix(m['avg_item_rating'].to_numpy().reshape(-1, 1))

    # normalize each block first, then apply block weights
    X_desc = normalize(X_desc, norm='l2', axis=1) * w_desc
    X_feat = normalize(X_feat, norm='l2', axis=1) * w_feat
    X_cat = normalize(X_cat, norm='l2', axis=1) * w_cat
    X_rating = normalize(X_rating, norm='l2', axis=1) * w_rating

    item_matrix = hstack([X_desc, X_feat, X_cat, X_rating]).tocsr()

    item_id_to_row = pd.Series(np.arange(len(m)), index=m['item_id']).to_dict()

    # Keep interactions that map to item rows
    interactions = interactions_df[interactions_df['item_id'].isin(item_id_to_row)].copy()
    user_ids = interactions['user_id'].unique()
    user_id_to_row = pd.Series(np.arange(len(user_ids)), index=user_ids).to_dict()

    interactions['item_row'] = interactions['item_id'].map(item_id_to_row).astype(np.int32)
    interactions['user_row'] = interactions['user_id'].map(user_id_to_row).astype(np.int32)

    R = csr_matrix(
        (
            interactions['rating'].values,
            (interactions['user_row'].values, interactions['item_row'].values)
        ),
        shape=(len(user_ids), len(m))
    )

    # Rating-weighted user profile with row normalization
    row_sums = np.array(R.sum(axis=1)).flatten()
    row_sums[row_sums == 0] = 1.0
    D_inv = diags(1.0 / row_sums)
    user_matrix = (D_inv @ R @ item_matrix).tocsr()

    artifacts = {
        'item_matrix': item_matrix,
        'user_matrix': user_matrix,
        'item_id_to_row': item_id_to_row,
        'user_id_to_row': user_id_to_row,
        'interactions': interactions,
        'vectorizers': (v_desc, v_feat, v_cat),
        'metadata_with_rating': m
    }
    return artifacts

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def compute_similarity(user_matrix, item_matrix, user_rows):
    U = user_matrix[user_rows]
    similarity = cosine_similarity(U, item_matrix)
    shape = similarity.shape
    return similarity, shape, similarity[0, :5]

def topk_metrics(eval_df, artifacts, k=10, relevance_threshold=3.0):
    item_matrix = artifacts['item_matrix']
    user_matrix = artifacts['user_matrix']
    item_id_to_row = artifacts['item_id_to_row']
    user_id_to_row = artifacts['user_id_to_row']
    interactions = artifacts['interactions']

    eval_df = eval_df[
        eval_df['user_id'].isin(user_id_to_row) & eval_df['item_id'].isin(item_id_to_row)
    ].copy()
    if len(eval_df) == 0:
        return {'users_evaluated': 0, 'precision_at_10': np.nan, 'map_at_10': np.nan, 'mrr_at_10': np.nan, 'hit_rate': np.nan, 'coverage': np.nan}

    eval_df['item_row'] = eval_df['item_id'].map(item_id_to_row)
    eval_users = np.array(eval_df['user_id'].unique())
    eval_user_rows = np.array([user_id_to_row[u] for u in eval_users], dtype=np.int32)

    relevant_df = eval_df[eval_df['rating'] >= relevance_threshold]
    relevant_by_user = relevant_df.groupby('user_id')['item_row'].apply(set).to_dict()

    seen_by_user = interactions.groupby('user_id')['item_row'].apply(set).to_dict()

    similarities, pred_shape, pred_sample = compute_similarity(user_matrix, item_matrix, eval_user_rows)
    print('shape:', pred_shape)
    print('pred_ratings sample:', pred_sample)

    topk_by_user = {}
    all_recommended_items = set()
    n_items = item_matrix.shape[0]

    for idx, u in enumerate(eval_users):
        sim = similarities[idx].copy()

        seen_items = seen_by_user.get(u, set())
        if seen_items:
            sim[list(seen_items)] = -np.inf

        finite_idx = np.where(np.isfinite(sim))[0]
        if finite_idx.size == 0:
            recs = np.array([], dtype=np.int32)
        else:
            kk = min(k, finite_idx.size)
            cand = np.argpartition(-sim[finite_idx], kk - 1)[:kk]
            recs = finite_idx[cand]
            recs = recs[np.argsort(-sim[recs])]

        topk_by_user[u] = recs
        all_recommended_items.update(recs.tolist())

    precisions, ap_scores, rr_scores, hit_rates = [], [], [], []

    for u in eval_users:
        recs = topk_by_user[u]
        rel = relevant_by_user.get(u, set())
        # hits if we are in relevant set or not for each position in the recommendation list
        hits = np.array([1 if r in rel else 0 for r in recs], dtype=np.int32)
        precisions.append(hits.sum() / k)
        hit_rates.append(1.0 if hits.sum() > 0 else 0.0)

        if len(rel) > 0:
            if hits.sum() > 0:
                cum_hits = np.cumsum(hits)
                hit_pos = np.where(hits == 1)[0]
                ap = (cum_hits[hit_pos] / (hit_pos + 1)).sum() / min(len(rel), k)
                rr = 1.0 / (hit_pos[0] + 1)
            else:
                ap, rr = 0.0, 0.0
            ap_scores.append(ap)
            rr_scores.append(rr)

    return {
        'users_evaluated': len(eval_users),
        'precision_at_10': float(np.mean(precisions)) if precisions else np.nan,
        'map_at_10': float(np.mean(ap_scores)) if ap_scores else np.nan,
        'mrr_at_10': float(np.mean(rr_scores)) if rr_scores else np.nan,
        'hit_rate': float(np.mean(hit_rates)) if hit_rates else np.nan,
        'coverage': float(len(all_recommended_items) / n_items) if n_items > 0 else np.nan
    }

## Hyperparameter Tuning (Small/Fast Search)
Primary objective: maximize MAP@10 on validation.  
Explain why you choose to maximize on MAP@10, because I think its the best metric. Mean average precision provides the best intel on how well a users recommended top 10 is performing. I am prioritizing ranking metrics over error based. Validate on MAP because it is more informative for ranking evaluation

In [24]:

# Small search space for laptop-friendly runtime
# tfidf hyperparameters for grid search 
# conduct a grid search with different hyperparameters for TF-IDF
# evaluate on validation and pick the best configuration based on Mean Average Precision at 10
tfidf_grid = {
    'min_df': [1, 2, 3, 5, 7, 10],
    'max_df': [0.75, 0.85, 0.95, 0.99],
    'ngram_range': [(1, 1), (1, 2)],
    'sublinear_tf': [True, False]
}
# explain these hyperparameters briefly in comments:
# min_df: minimum document frequency for a term to be included in the vocabulary (filters out rare terms)
# max_df: maximum document frequency for a term to be included (filters out very common terms)
# ngram_range: whether to include unigrams only (1,1) or also bigrams (1,2)
# sublinear_tf: whether to apply sublinear scaling to term frequencies (replace tf with 1 + log(tf)) which can help with very common terms

# block weights for description, features, categories, ratings
# explain what these weights mean in comments:
# these weights control the relative importance of each block of features in the final item representation.
# for example, (1.0, 1.0, 1.0, 0.1) means we give equal weight to description, features, and categories, 
# but much less weight to ratings
weight_grid = [
    (1.0, 1.0, 1.0, 0.1),
    (1.5, 1.0, 1.0, 0.1),
    (1.0, 1.5, 1.0, 0.3),
    (1.0, 1.0, 1.5, 0.3),
    (1.5, 1.0, 0.75, 0.1),
    (1.5, 1.0, 0.5, 0.1),
    (1.5, 1.0, 0.25, 0.1),
    (1.5, 1.0, 0.1, 0.1)
]

all_cfgs = list(product(
    tfidf_grid['min_df'],
    tfidf_grid['max_df'],
    tfidf_grid['ngram_range'],
    tfidf_grid['sublinear_tf']
))

random.seed(42)
random.shuffle(all_cfgs)
all_cfgs = all_cfgs[:20]  # small budget

results = []
for cfg in all_cfgs:
    tfidf_cfg = {
        'min_df': cfg[0],
        'max_df': cfg[1],
        'ngram_range': cfg[2],
        'sublinear_tf': cfg[3]
    }

    for weights in weight_grid:
        # Build matrices and evaluate on validation set
        artifacts = build_matrices(metadata_df, sub_train_df, tfidf_cfg, weights)
        val_metrics = topk_metrics(val_df, artifacts, k=10, relevance_threshold=3.0)

        results.append({
            'map_at_10': val_metrics['map_at_10'],
            'precision_at_10': val_metrics['precision_at_10'],
            'mrr_at_10': val_metrics['mrr_at_10'],
            'hit_rate': val_metrics['hit_rate'],
            'coverage': val_metrics['coverage'],
            'users_evaluated': val_metrics['users_evaluated'],
            'tfidf_cfg': tfidf_cfg,
            'weights': weights
        })

# sorting by MAP could be a trade off between relevance and coverage
results_df = pd.DataFrame(results).sort_values(by='map_at_10', ascending=False).reset_index(drop=True)
results_df.head(10)

shape: (1387, 927)
pred_ratings sample: [0.50393047 0.50227374 0.57608785 0.5929108  0.46845408]
shape: (1387, 927)
pred_ratings sample: [0.44308551 0.43049812 0.46562413 0.49286223 0.37526266]
shape: (1387, 927)
pred_ratings sample: [0.42696907 0.43481813 0.55191873 0.56611143 0.40744243]
shape: (1387, 927)
pred_ratings sample: [0.64232672 0.64134886 0.70988157 0.71913102 0.6213872 ]
shape: (1387, 927)
pred_ratings sample: [0.36811495 0.35281004 0.37778817 0.41150393 0.28564964]
shape: (1387, 927)
pred_ratings sample: [0.29346013 0.27522003 0.28574743 0.32657834 0.19517957]
shape: (1387, 927)
pred_ratings sample: [0.23453314 0.21379768 0.21005488 0.25700248 0.12280722]
shape: (1387, 927)
pred_ratings sample: [0.21528445 0.19369748 0.18477842 0.23382377 0.09897044]
shape: (1387, 927)
pred_ratings sample: [0.44986529 0.44606207 0.51327619 0.53405292 0.43765185]
shape: (1387, 927)
pred_ratings sample: [0.38080192 0.36920699 0.39564325 0.42914859 0.35286917]
shape: (1387, 927)
pred_rating

,map_at_10,precision_at_10,mrr_at_10,hit_rate,coverage,users_evaluated,tfidf_cfg,weights
0,0.022887,0.006561,0.022887,0.065609,0.706580,1387,"{'min_df': 5, 'max_df': 0.99, 'ngram_range': (...","(1.5, 1.0, 0.5, 0.1)"
1,0.022887,0.006561,0.022887,0.065609,0.706580,1387,"{'min_df': 5, 'max_df': 0.75, 'ngram_range': (...","(1.5, 1.0, 0.5, 0.1)"
2,0.022887,0.006561,0.022887,0.065609,0.706580,1387,"{'min_df': 5, 'max_df': 0.95, 'ngram_range': (...","(1.5, 1.0, 0.5, 0.1)"
3,0.022301,0.005984,0.022301,0.059841,0.687163,1387,"{'min_df': 5, 'max_df': 0.99, 'ngram_range': (...","(1.5, 1.0, 0.75, 0.1)"
4,0.022301,0.005984,0.022301,0.059841,0.687163,1387,"{'min_df': 5, 'max_df': 0.75, 'ngram_range': (...","(1.5, 1.0, 0.75, 0.1)"
5,0.022301,0.005984,0.022301,0.059841,0.687163,1387,"{'min_df': 5, 'max_df': 0.95, 'ngram_range': (...","(1.5, 1.0, 0.75, 0.1)"
6,0.021994,0.006273,0.021994,0.062725,0.673139,1387,"{'min_df': 3, 'max_df': 0.75, 'ngram_range': (...","(1.5, 1.0, 0.5, 0.1)"
7,0.021994,0.006273,0.021994,0.062725,0.673139,1387,"{'min_df': 3, 'max_df': 0.95, 'ngram_range': (...","(1.5, 1.0, 0.5, 0.1)"
8,0.021940,0.006128,0.021940,0.061283,0.661273,1387,"{'min_df': 7, 'max_df': 0.95, 'ngram_range': (...","(1.5, 1.0, 0.5, 0.1)"
9,0.021400,0.006056,0.021400,0.060562,0.676375,1387,"{'min_df': 5, 'max_df': 0.75, 'ngram_range': (...","(1.5, 1.0, 0.25, 0.1)"


In [27]:
best_row = results_df.iloc[0]
best_tfidf_cfg = best_row['tfidf_cfg']
best_weights = best_row['weights']

print('Best validation MAP@10:', round(float(best_row['map_at_10']), 6))
print('Best TF-IDF config:', best_tfidf_cfg)
print('Best block weights (desc, feat, cat, rating):', best_weights)
print('Validation Precision@10:', round(float(best_row['precision_at_10']), 6))
print('Validation MRR@10:', round(float(best_row['mrr_at_10']), 6))
print('Validation HitRate@10:', round(float(best_row['hit_rate']), 6))
print('Validation Coverage:', round(float(best_row['coverage']), 6))

Best validation MAP@10: 0.022887
Best TF-IDF config: {'min_df': 5, 'max_df': 0.99, 'ngram_range': (1, 2), 'sublinear_tf': False}
Best block weights (desc, feat, cat, rating): (1.5, 1.0, 0.5, 0.1)
Validation Precision@10: 0.006561
Validation MRR@10: 0.022887
Validation HitRate@10: 0.065609
Validation Coverage: 0.70658


## Final Model (Retrain on Full Train with Best Config)
retrain with full train interactions and evaluate once on test.

In [28]:
final_artifacts = build_matrices(metadata_df, train_df, best_tfidf_cfg, best_weights)

print('Final item matrix shape:', final_artifacts['item_matrix'].shape)
print('Final user matrix shape:', final_artifacts['user_matrix'].shape)
print('Users represented:', len(final_artifacts['user_id_to_row']))

Final item matrix shape: (927, 8056)
Final user matrix shape: (1389, 8056)
Users represented: 1389


In [29]:
# save the model
import pickle
with open('models/best_content_based_model.pkl', 'wb') as f:
    pickle.dump(final_artifacts, f)


In [30]:
# Test ranking metrics
test_metrics = topk_metrics(test_df, final_artifacts, k=10, relevance_threshold=3.0)

print('Content-based system (tuned on validation)')
print('Users evaluated:', test_metrics['users_evaluated'])
print(f"Precision@10: {test_metrics['precision_at_10']:.4f}")
print(f"MAP@10:       {test_metrics['map_at_10']:.4f}")
print(f"MRR@10:       {test_metrics['mrr_at_10']:.4f}")
print(f"Hit Rate:     {test_metrics['hit_rate']:.4f}")
print(f"Coverage:     {test_metrics['coverage']:.4f}")

shape: (1389, 927)
pred_ratings sample: [0.1211641  0.13775061 0.12649271 0.13199884 0.07919554]
Content-based system (tuned on validation)
Users evaluated: 1389
Precision@10: 0.0266
MAP@10:       0.0222
MRR@10:       0.0794
Hit Rate:     0.2268
Coverage:     0.6947


In [31]:
# Optional: test RMSE/MAE on known test pairs
def predict_rating_cosine(user_id, item_id, artifacts, min_rating=1.0, max_rating=5.0):
    user_id_to_row = artifacts['user_id_to_row']
    item_id_to_row = artifacts['item_id_to_row']
    user_matrix = artifacts['user_matrix']
    item_matrix = artifacts['item_matrix']

    u = user_id_to_row.get(user_id)
    i = item_id_to_row.get(item_id)
    if u is None or i is None:
        return np.nan

    u_vec = user_matrix[u]
    i_vec = item_matrix[i]

    dot_ui = u_vec.multiply(i_vec).sum()
    u_norm = np.sqrt(u_vec.multiply(u_vec).sum())
    i_norm = np.sqrt(i_vec.multiply(i_vec).sum())

    if u_norm == 0 or i_norm == 0:
        return np.nan

    cos_sim = float(dot_ui / (u_norm * i_norm))
    cos_sim = np.clip(cos_sim, 0.0, 1.0)
    pred_rating = min_rating + (max_rating - min_rating) * cos_sim
    return pred_rating

eval_pairs = test_df[
    test_df['user_id'].isin(final_artifacts['user_id_to_row']) &
    test_df['item_id'].isin(final_artifacts['item_id_to_row'])
].copy()

eval_pairs['pred_rating_cosine'] = eval_pairs.apply(
    lambda r: predict_rating_cosine(r['user_id'], r['item_id'], final_artifacts),
    axis=1
)

valid_preds = eval_pairs.dropna(subset=['pred_rating_cosine'])
rmse_test = np.sqrt(mean_squared_error(valid_preds['rating'], valid_preds['pred_rating_cosine']))
mae_test = mean_absolute_error(valid_preds['rating'], valid_preds['pred_rating_cosine'])

print('Known test pairs used for RMSE/MAE:', len(valid_preds))
print(f'TEST RMSE: {rmse_test:.4f}, TEST MAE: {mae_test:.4f}')

Known test pairs used for RMSE/MAE: 6645
TEST RMSE: 2.7872, TEST MAE: 2.6028


## Report Notes
- Item subset only items that appear in both train and test.
- Hyperparameters were selected on a validation split derived from train only.
- Primary tuning objective was MAP@10.
- Final test results were computed once with the locked best configuration.
- Feature blocks were individually normalized and then weighted before concatenation.

Suggested ablations to include in report:
1. Remove avg_item_rating block and compare MAP@10 / Coverage.
2. Uniform weights vs tuned weights.
3. Unigrams vs unigrams+bigrams.
4. min_df/max_df sensitivity.